# MRC Combiner Output Headroom

"
"Investigates the actual fixed-point output amplitude of Trouper's live `mrc_combiner.v` path.

"
"This notebook answers a different question from `07_mrc_weight_quantisation.ipynb`:
"
"- `07_...` asks whether int8 weight quantisation hurts MRC gain.
"
"- this notebook asks whether the raw 8-bit combiner arithmetic clips before the `sd_remod` input.

"
"**Key question:** if firmware scales the strongest branch weight to a value like `45`, does that keep the live RTL output within range?

"
"**Short answer:** no. In the current RTL, `45` is far too large for linear no-clipping MRC unless the branch samples are extremely small.

"
"**Relevant RTL path:**
"
"- `trouper_top.v`: only the high byte of each 16-bit weight word is fed to `mrc_combiner`
"
"- `mrc_combiner.v`: 8-bit weights x 8-bit samples, sum 4 complex products, arithmetic `>>> 1`, optional `COMB_POST_GAIN_SHIFT`, then saturate to int8
"
"- `trouper_top.v`: remodulator input is then `comb_y >>> REMOD_BACKOFF_SHIFT`
"
"- reset default is `COMB_POST_GAIN_SHIFT=0`, `REMOD_BACKOFF_SHIFT=1`


In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

NR = 4
FULL_SCALE = 127
SAFE_REMOD = 90   # about -3 dBFS target used in planning docs
PLOT_DIR = pathlib.Path('../plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)


---
## 1  RTL Model

"
"For one output sample, the live datapath is:

"
"$$y = \mathrm{sat8}\!\left(\left(\sum_{k=0}^{3} W_k x_kight) \gg 1\; \ll\; 	exttt{COMB\_POST\_GAIN}ight)$$

"
"and the re-modulator sees:

"
"$$y_{\mathrm{remod}} = y \gg 	exttt{REMOD\_BACKOFF\_SHIFT}$$

"
"There is no fractional rescale after the multiply. The combiner consumes plain signed 8-bit integers.

"
"For matched MRC with branch samples $x_k = a_k e^{j\phi_k}$ and firmware weights proportional to $W_k = \alpha a_k e^{-j\phi_k}$, the complex phase cancels and the pre-saturation combiner output becomes purely real:

"
"$$y_{\mathrm{pre}} = \frac{1}{2}\,\alpha\,\sum_k a_k^2$$

"
"If the strongest branch has amplitude $A = \max_k a_k$ and firmware normalises the strongest branch weight magnitude to `W_MAX`, then $\alpha = W_{\max}/A$ and:

"
"$$y_{\mathrm{pre}} = \frac{1}{2}\,A\,W_{\max}\,\sum_k r_k^2\qquad r_k = a_k/A$$

"
"This is the quantity that must stay within `[-127, 127]` to avoid combiner clipping.


In [ ]:
def arshift(val: int, shift: int) -> int:
    """Arithmetic right shift for Python ints."""
    return val >> shift


def rtl_complex_mac(x: np.ndarray, w: np.ndarray, post_gain_shift: int = 0, remod_backoff_shift: int = 1):
    """Exact integer model of the relevant mrc_combiner + remod input path.

    x, w: complex arrays whose real/imag parts are signed integers.
    """
    x_i = np.asarray(np.real(x), dtype=int)
    x_q = np.asarray(np.imag(x), dtype=int)
    w_i = np.asarray(np.real(w), dtype=int)
    w_q = np.asarray(np.imag(w), dtype=int)

    acc_i = int(np.sum(w_i * x_i - w_q * x_q))
    acc_q = int(np.sum(w_i * x_q + w_q * x_i))

    guarded_i = arshift(acc_i, 1)
    guarded_q = arshift(acc_q, 1)
    shifted_i = guarded_i << post_gain_shift
    shifted_q = guarded_q << post_gain_shift

    comb_i = int(np.clip(shifted_i, -128, 127))
    comb_q = int(np.clip(shifted_q, -128, 127))
    remod_i = arshift(comb_i, remod_backoff_shift)
    remod_q = arshift(comb_q, remod_backoff_shift)

    return {
        'acc_i': acc_i, 'acc_q': acc_q,
        'guarded_i': guarded_i, 'guarded_q': guarded_q,
        'comb_i': comb_i, 'comb_q': comb_q,
        'remod_i': remod_i, 'remod_q': remod_q,
        'comb_clipped': (comb_i != shifted_i) or (comb_q != shifted_q),
        'remod_safe': max(abs(remod_i), abs(remod_q)) < SAFE_REMOD,
    }


def coherent_profile(strong_amp: float, spread_dB: float, nr: int = NR) -> np.ndarray:
    """Branch amplitudes for SNRs [0, -d/3, ..., -d] dB relative to strongest.

    SNR is proportional to amplitude squared, so amplitude ratio is 10^(-d/20).
    """
    return strong_amp * 10.0 ** (-np.linspace(0.0, spread_dB, nr) / 20.0)


def pre_sat_peak(strong_amp: float, w_max: float, spread_dB: float, nr: int = NR) -> float:
    amps = coherent_profile(strong_amp, spread_dB, nr)
    ratios = amps / np.max(amps)
    return 0.5 * strong_amp * w_max * np.sum(ratios ** 2)


def max_safe_wmax(strong_amp: float, spread_dB: float, nr: int = NR, comb_limit: int = FULL_SCALE) -> float:
    amps = coherent_profile(strong_amp, spread_dB, nr)
    ratios = amps / np.max(amps)
    denom = strong_amp * np.sum(ratios ** 2)
    return (2.0 * comb_limit / denom) if denom > 0 else np.inf


---
## 2  Analytic Bound

"
"To avoid combiner clipping with `COMB_POST_GAIN_SHIFT=0`, the matched-MRC coherent-sum bound is:

"
"$$\frac{1}{2} A W_{\max} \sum_k r_k^2 \le 127$$

"
"so the strongest-branch weight scale must satisfy:

"
"$$W_{\max} \le \frac{254}{A \sum_k r_k^2}$$

"
"This bound is much tighter than the weight-quantisation notebook because that earlier notebook intentionally ignored absolute amplitude.


In [ ]:
branch_amps = np.array([90, 64, 45, 32, 16, 8, 4], dtype=float)
spreads = [0, 6, 12, 18]

print(f"{'A_strong':>8}  {'spread':>8}  {'max safe W_MAX':>14}  {'equal-branch safe weight':>24}")
print('-' * 64)
for A in branch_amps:
    for spread in spreads:
        wmax = max_safe_wmax(A, spread)
        equal_safe = 254 / (NR * A)
        print(f"{A:8.1f}  {spread:8.1f}  {wmax:14.3f}  {equal_safe:24.3f}")


In [ ]:
amps = np.linspace(4, 100, 193)
spreads_dB = np.linspace(0, 20, 81)
WMAXS = [45, 15, 4, 1]

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
for ax, w_max in zip(axes.ravel(), WMAXS):
    Z = np.array([[pre_sat_peak(A, w_max, spread) for spread in spreads_dB] for A in amps])
    im = ax.contourf(spreads_dB, amps, Z, levels=[0, 32, 64, 90, 127, 256, 512, 2048, 8192], cmap='viridis')
    ax.contour(spreads_dB, amps, Z, levels=[127], colors='r', linewidths=2)
    ax.set_title(f'Pre-sat |y| for W_MAX={w_max}')
    ax.set_xlabel('SNR spread d (dB)')
    ax.set_ylabel('Strongest branch amplitude A (counts)')
    ax.grid(True, alpha=0.2)

fig.colorbar(im, ax=axes.ravel().tolist(), label='Pre-saturation combiner magnitude |y_pre| (counts)')
fig.suptitle('Combiner headroom before int8 saturation', fontweight='bold')
fig.tight_layout()
plt.savefig(PLOT_DIR / 'mrc_output_headroom_regions.png', bbox_inches='tight')
plt.show()
print('Saved: sim/plots/mrc_output_headroom_regions.png')


---
## 3  Concrete Scenarios

"
"The next table evaluates a few representative coherent equal-phase cases.

"
"Each case uses real-positive branch samples and real-positive matched weights, which is the worst-case amplitude condition for clipping and exactly the condition MRC tries to create after phase alignment.


In [ ]:
def equal_branch_case(branch_amp: int, branch_weight: int, nr: int = NR, post_gain_shift: int = 0, remod_backoff_shift: int = 1):
    x = np.array([branch_amp + 0j] * nr)
    w = np.array([branch_weight + 0j] * nr)
    res = rtl_complex_mac(x, w, post_gain_shift=post_gain_shift, remod_backoff_shift=remod_backoff_shift)
    return {
        'A': branch_amp,
        'W': branch_weight,
        'acc_i': res['acc_i'],
        'guarded_i': res['guarded_i'],
        'comb_i': res['comb_i'],
        'remod_i': res['remod_i'],
        'comb_clipped': res['comb_clipped'],
        'remod_safe': res['remod_safe'],
    }

cases = [
    equal_branch_case(90, 45),
    equal_branch_case(90,  1),
    equal_branch_case(32,  1),
    equal_branch_case(16,  3),
    equal_branch_case( 8,  7),
]

print(f"{'A':>4}  {'W':>4}  {'acc':>8}  {'>>>1':>8}  {'comb_y':>8}  {'remod_in':>10}  {'comb_clipped':>13}  {'remod_safe':>11}")
print('-' * 86)
for c in cases:
    print(f"{c['A']:>4}  {c['W']:>4}  {c['acc_i']:>8}  {c['guarded_i']:>8}  {c['comb_i']:>8}  {c['remod_i']:>10}  {str(c['comb_clipped']):>13}  {str(c['remod_safe']):>11}")


In [ ]:
A_vals = np.array([90, 64, 45, 32, 16, 8, 4])
req_equal_weight = 254 / (NR * A_vals)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(A_vals, req_equal_weight, 'o-', lw=2, label='Max equal branch weight (no combiner clip)')
ax.axhline(45, color='tab:red', ls='--', lw=1.5, label='W = 45 reference')
ax.axhline(1, color='tab:green', ls=':', lw=1.5, label='W = 1')
ax.set_xlabel('Per-branch amplitude A (counts)')
ax.set_ylabel('Largest equal branch weight allowed')
ax.set_title('Equal-strength 4-branch case: no-clipping weight limit')
ax.invert_xaxis()
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
plt.savefig(PLOT_DIR / 'mrc_output_examples.png', bbox_inches='tight')
plt.show()
print('Saved: sim/plots/mrc_output_examples.png')


---
## 4  Remodulator Safety vs Combiner Clipping

"
"These are separate questions.

"
"Because `trouper_top.v` applies `remod_in = comb_y >>> REMOD_BACKOFF_SHIFT`, the default reset setting `REMOD_BACKOFF_SHIFT=1` guarantees the remodulator sees at most `63` counts on the positive side and `64` counts on the negative side, even if the combiner has already saturated to int8.

"
"So with the current live RTL:
"
"- `REMOD_BACKOFF_SHIFT=1` protects `sd_remod` from overdrive
"
"- but it does not prevent the combiner from clipping first
"
"- therefore a large `W_MAX` can still destroy linear MRC behavior even though the remodulator itself remains safe


In [ ]:
for shift in [0, 1, 2, 3]:
    pos = 127 >> shift
    neg = (-128) >> shift
    safe = max(abs(pos), abs(neg)) < SAFE_REMOD
    print(f'REMOD_BACKOFF_SHIFT={shift}: remod range [{neg}, {pos}], safe_vs_90={safe}')


---
## 5  Conclusions

Key findings:

1. **`W_MAX=45` is not a safe live-hardware amplitude rule.** With four equal branches at `90` counts, `W=45` gives a guarded combiner value of `8100`, so the combiner rails at `127` long before the remodulator backoff stage.

2. **No-clipping linear MRC is very restrictive in the current raw-int8 combiner.** At `90` counts per branch and four equal branches, the equal-weight limit is only `0.705` per branch, so even `W=1` already clips. Linear operation at that branch amplitude therefore requires either lower input amplitude, unequal branch strengths, or effectively sub-LSB weight scaling that the current raw-int8 datapath cannot express directly.

3. **The default remod backoff solves a different problem.** `REMOD_BACKOFF_SHIFT=1` makes the remodulator input safe (`<= 63` counts) even if the combiner output has already clipped. So remod safety is protected, but linear combining quality is not.

4. **This exposes a spec/implementation tension.** Planning text that talks about Q1.15-normalised weights and per-branch `<= 90` counts being sufficient for the MRC output to fit after `>>>1` is not consistent with the current RTL wiring that uses only the high byte as a raw 8-bit multiplier operand.

5. **Decision-useful implication.** If firmware owns weights, it should treat `W_MAX=45` only as a quantisation-study parameter, not as a live normalization target. A real firmware rule must be derived from the combiner amplitude bound, or the architecture must accept combiner clipping as the normal path.
